## Import libraries

In [ ]:
%matplotlib inline
import sys
import torch
import matplotlib.pyplot as plt
import pandas as pd
import yaml

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

try:
    import ultralytics
    print(f"Ultralytics version: {ultralytics.__version__}")
except ImportError:
    print("Ultralytics not installed")

try:
    import cv2
    print(f"OpenCV version: {cv2.__version__}")
except ImportError:
    print("OpenCV not installed")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Load model

In [ ]:
from ultralytics import YOLO
import os
dataset_path = os.environ.get("RDD_DATASET_PATH", "data/RDD_SPLIT")
print(f"Dataset path: {dataset_path}")
model = YOLO("yolo11n.pt")
model.to(device)
print("Pretrained YOLOv11 loaded.")

In [ ]:
# Fine-tuning YOLOv11 on the road defect dataset

import yaml

# Keep the class names in one place so the dataset labels and YOLO config stay aligned.
CLASS_NAMES = [
    "longitudinal_crack",
    "transverse_crack",
    "alligator_crack",
    "other_corruption",
    "pothole",
]

data_yaml = {
    "path": dataset_path,
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
}

with open("data.yaml", "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)


In [ ]:
import os
import random
import shutil
from collections import defaultdict

random.seed(42)


def get_dominant_class(label_path):
    """Return the most frequent class id in a YOLO label file, or None if empty."""
    with open(label_path) as f:
        classes = [int(line.split()[0]) for line in f if line.strip()]
    return max(set(classes), key=classes.count) if classes else None


def group_by_dominant_class(labels_dir):
    """Return {class_id: [image_stem, ...]} for every label file in labels_dir."""
    by_class = defaultdict(list)
    for fname in os.listdir(labels_dir):
        if not fname.endswith(".txt"):
            continue
        cls = get_dominant_class(os.path.join(labels_dir, fname))
        if cls is not None:
            by_class[cls].append(fname[:-4])  # strip .txt
    return by_class


def sample_equal(by_class, subset_size):
    """Equal quota per class, regardless of original frequency."""
    quota = subset_size // len(by_class)
    stems = []
    for cls, cls_stems in sorted(by_class.items()):
        take = min(quota, len(cls_stems))
        if take < quota:
            print(f"    WARNING: {CLASS_NAMES[cls]} only has {len(cls_stems)} images, "
                  f"short of the {quota} quota -- using all of them.")
        stems.extend(random.sample(cls_stems, take))
    return list(dict.fromkeys(stems))


def sample_original_ratio(by_class, subset_size):
    """Sample proportionally to each class's original frequency."""
    total = sum(len(v) for v in by_class.values())
    stems = []
    for cls, cls_stems in sorted(by_class.items()):
        quota = round(subset_size * len(cls_stems) / total)
        take = min(quota, len(cls_stems))
        stems.extend(random.sample(cls_stems, take))
    return list(dict.fromkeys(stems))


def copy_subset(split, stems, subset_name):
    """Copy the sampled images + labels for `split` into dataset_path/subset_name/."""
    images_dir = os.path.join(dataset_path, split, "images")
    labels_dir = os.path.join(dataset_path, split, "labels")
    out_images_dir = os.path.join(dataset_path, subset_name, "images")
    out_labels_dir = os.path.join(dataset_path, subset_name, "labels")
    os.makedirs(out_images_dir, exist_ok=True)
    os.makedirs(out_labels_dir, exist_ok=True)

    sample_img = next(f for f in os.listdir(images_dir) if not f.startswith("."))
    img_ext = os.path.splitext(sample_img)[1]

    for stem in stems:
        shutil.copy(os.path.join(images_dir, stem + img_ext), os.path.join(out_images_dir, stem + img_ext))
        shutil.copy(os.path.join(labels_dir, stem + ".txt"), os.path.join(out_labels_dir, stem + ".txt"))

    print(f"  Copied {len(stems)} images -> {out_images_dir}")

In [ ]:
# Target sizes per split (adjust as needed)
subset_sizes = {"train": 2687, "val": 576, "test": 576}

for split, subset_size in subset_sizes.items():
    labels_dir = os.path.join(dataset_path, split, "labels")
    by_class = group_by_dominant_class(labels_dir)

    print(f"\n{'='*50}\n{split.upper()} -- full distribution\n{'='*50}")
    for cls, stems in sorted(by_class.items()):
        print(f"  {CLASS_NAMES[cls]}: {len(stems)} images")

    print(f"\n{split}: building EQUAL-ratio subset (target {subset_size})")
    equal_stems = sample_equal(by_class, subset_size)
    copy_subset(split, equal_stems, f"{split}_subset_equal")

    print(f"\n{split}: building ORIGINAL-ratio subset (target {subset_size})")
    original_stems = sample_original_ratio(by_class, subset_size)
    copy_subset(split, original_stems, f"{split}_subset_original")

## Load dataset

In [ ]:

data_yaml = {
    "path": dataset_path,
    "train": "train_subset_equal/images",
    "val": "val_subset_equal/images",
    "test": "test_subset_equal/images",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
}

with open("data.yaml", "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("data.yaml written:")
print(yaml.dump(data_yaml, sort_keys=False))

## Train model

In [ ]:
#Training (1)

train_results = model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=20,
    lr0=0.0001,
    optimizer="SGD",
    augment=True,
    project="road_defect_runs",
    name="yolov11_baseline_50_16_0.0001",
    exist_ok=True,
)

print("Training complete.")

In [ ]:
#Training (2)

train_results = model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=20,
    lr0=0.001,
    optimizer="SGD",
    augment=True,
    project="road_defect_runs",
    name="yolov11_baseline_50_16_0.001",
    exist_ok=True,
)

print("Training complete.")

In [ ]:
# Training (3)
train_results = model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=20,
    lr0=0.01,
    optimizer="SGD",
    augment=True,
    project="road_defect_runs",
    name="yolov11_baseline_50_16_0.01",
    exist_ok=True,
)

print("Training complete.")

## Test results

In [ ]:
# Testing (1)

project="road_defect_runs"
name="yolov11_baseline_50_16_0.0001"

In [ ]:
# Testing (1)

# Ultralytics saves weights under runs/detect/<project>/<name>/weights/ by default
weights_path = f"runs/detect/{project}/{name}/weights/best.pt"

test_model = YOLO(weights_path)
test_model.to(device)

imgsz = 640

test_metrics = test_model.val(data="data.yaml", imgsz=imgsz, split="test")

print(f"Test mAP50:    {test_metrics.box.map50:.4f}")
print(f"Test mAP50-95: {test_metrics.box.map:.4f}")
print(f"Test precision: {test_metrics.box.mp:.4f}")
print(f"Test recall:    {test_metrics.box.mr:.4f}")

In [ ]:
# Testing (2)

project = "road_defect_runs"
name = "yolov11_baseline_50_16_0.001"

In [ ]:
# Testing (2)

# Ultralytics saves weights under runs/detect/<project>/<name>/weights/ by default
weights_path = f"runs/detect/{project}/{name}/weights/best.pt"

test_model = YOLO(weights_path)
test_model.to(device)

imgsz = 640

test_metrics = test_model.val(data="data.yaml", imgsz=imgsz, split="test")

print(f"Test mAP50:    {test_metrics.box.map50:.4f}")
print(f"Test mAP50-95: {test_metrics.box.map:.4f}")
print(f"Test precision: {test_metrics.box.mp:.4f}")
print(f"Test recall:    {test_metrics.box.mr:.4f}")

In [ ]:
# Testing (3)

project="road_defect_runs"
name="yolov11_baseline_50_16_0.01"

In [ ]:
# Testing (3)

# Ultralytics saves weights under runs/detect/<project>/<name>/weights/ by default
weights_path = f"runs/detect/{project}/{name}/weights/best.pt"

test_model = YOLO(weights_path)
test_model.to(device)

imgsz = 640

test_metrics = test_model.val(data="data.yaml", imgsz=imgsz, split="test")

print(f"Test mAP50:    {test_metrics.box.map50:.4f}")
print(f"Test mAP50-95: {test_metrics.box.map:.4f}")
print(f"Test precision: {test_metrics.box.mp:.4f}")
print(f"Test recall:    {test_metrics.box.mr:.4f}")

## Plot results

In [ ]:
# Plot (1)

project="road_defect_runs"
name="yolov11_baseline_50_16_0.0001"

In [ ]:
# Plot (1)

results_csv = f"runs/detect/{project}/{name}/results.csv"
results_history = pd.read_csv(results_csv)
results_history.columns = [c.strip() for c in results_history.columns]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(results_history["epoch"], results_history["train/box_loss"], label="train")
axes[0].plot(results_history["epoch"], results_history["val/box_loss"], label="val")
axes[0].set_title("Box loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(results_history["epoch"], results_history["metrics/mAP50(B)"], label="mAP50")
axes[1].plot(results_history["epoch"], results_history["metrics/mAP50-95(B)"], label="mAP50-95")
axes[1].set_title("Validation mAP")
axes[1].set_xlabel("epoch")
axes[1].legend()

axes[2].plot(results_history["epoch"], results_history["metrics/precision(B)"], label="precision")
axes[2].plot(results_history["epoch"], results_history["metrics/recall(B)"], label="recall")
axes[2].set_title("Precision / Recall")
axes[2].set_xlabel("epoch")
axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

In [ ]:
# Plot (2)

project = "road_defect_runs"
name = "yolov11_baseline_50_16_0.001"

In [ ]:
# Plot (2)

results_csv = f"runs/detect/{project}/{name}/results.csv"
results_history = pd.read_csv(results_csv)
results_history.columns = [c.strip() for c in results_history.columns]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(results_history["epoch"], results_history["train/box_loss"], label="train")
axes[0].plot(results_history["epoch"], results_history["val/box_loss"], label="val")
axes[0].set_title("Box loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(results_history["epoch"], results_history["metrics/mAP50(B)"], label="mAP50")
axes[1].plot(results_history["epoch"], results_history["metrics/mAP50-95(B)"], label="mAP50-95")
axes[1].set_title("Validation mAP")
axes[1].set_xlabel("epoch")
axes[1].legend()

axes[2].plot(results_history["epoch"], results_history["metrics/precision(B)"], label="precision")
axes[2].plot(results_history["epoch"], results_history["metrics/recall(B)"], label="recall")
axes[2].set_title("Precision / Recall")
axes[2].set_xlabel("epoch")
axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

In [ ]:
# Plot 3

project = "road_defect_runs"
name = "yolov11_baseline_50_16_0.01"

In [ ]:
# Plot (3)

results_csv = f"runs/detect/{project}/{name}/results.csv"
results_history = pd.read_csv(results_csv)
results_history.columns = [c.strip() for c in results_history.columns]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(results_history["epoch"], results_history["train/box_loss"], label="train")
axes[0].plot(results_history["epoch"], results_history["val/box_loss"], label="val")
axes[0].set_title("Box loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(results_history["epoch"], results_history["metrics/mAP50(B)"], label="mAP50")
axes[1].plot(results_history["epoch"], results_history["metrics/mAP50-95(B)"], label="mAP50-95")
axes[1].set_title("Validation mAP")
axes[1].set_xlabel("epoch")
axes[1].legend()

axes[2].plot(results_history["epoch"], results_history["metrics/precision(B)"], label="precision")
axes[2].plot(results_history["epoch"], results_history["metrics/recall(B)"], label="recall")
axes[2].set_title("Precision / Recall")
axes[2].set_xlabel("epoch")
axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

## Camera Detection


In [ ]:
import time
import cv2

webcam_model = YOLO(weights_path)
webcam_model.to(device)

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Unable to access webcam.")
else:
    try:
        prev_time = time.time()
        while True:
            ret, frame = cap.read()
            if not ret:
                print("Failed to read frame from webcam.")
                break

            results = webcam_model(frame, conf=0.25, verbose=False)
            result = results[0]
            annotated_frame = result.plot()

            current_time = time.time()
            fps = 1.0 / max(current_time - prev_time, 1e-6)
            prev_time = current_time
            num_defects = 0 if result.boxes is None else len(result.boxes)

            cv2.putText(
                annotated_frame,
                f"Defects: {num_defects} | FPS: {fps:.1f}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2,
            )
            cv2.putText(
                annotated_frame,
                "Press 'q' to quit",
                (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2,
            )

            cv2.imshow("Road Defect Detection", annotated_frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        # Always release the webcam and close windows, even if interrupted mid-loop
        cap.release()
        cv2.destroyAllWindows()
        cv2.waitKey(1)